# Model overview

Visual summary of model size, semantic content, and diagnostics.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

Loaded 42 SysML files from /Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model


In [2]:
from IPython.display import HTML, display

semantic_types = [
    'Package', 'PartDefinition', 'PartUsage', 'RequirementDefinition',
    'RequirementUsage', 'ActionDefinition', 'ActionUsage', 'PortDefinition',
    'PortUsage', 'ConnectionDefinition', 'ConnectionUsage', 'StateDefinition',
    'StateUsage', 'AttributeDefinition', 'AttributeUsage'
]
counts = {}
for type_name in semantic_types:
    node_type = getattr(syside, type_name, None)
    if node_type is not None:
        counts[type_name] = len(list(model.elements(node_type)))

largest = max(counts.values(), default=1)
rows = ''.join(
    f'<tr><td>{name}</td><td>{count:,}</td><td><div style="background:#2d8d70;height:12px;width:{max(2, count * 320 / largest):.0f}px"></div></td></tr>'
    for name, count in sorted(counts.items(), key=lambda item: item[1], reverse=True)
)
display(HTML(f'<h3>Semantic inventory</h3><table><tr><th>Element type</th><th>Count</th><th>Relative size</th></tr>{rows}</table>'))

{
    'SysML files': len(SYSML_FILES),
    'Errors': len(list(diagnostics.errors)),
    'Warnings': len(list(diagnostics.warnings)),
    'Information': len(list(diagnostics.infos)),
}

Element type,Count,Relative size
AttributeUsage,"5,056",
ConnectionUsage,691,
PartUsage,425,
RequirementUsage,105,
ActionUsage,44,
Package,42,
StateUsage,10,
ActionDefinition,6,
PortDefinition,1,
PartDefinition,0,


{'SysML files': 42, 'Errors': 8009, 'Warnings': 0, 'Information': 0}